<hr style="border: 6px solid#003262;" />

<div align="center">
    <img src="images/thumbnails/ehcastroh_teach_banner_flower.png" align="center" width="20%">
</div>

<br>

# EXPLORATORY DATA ANALYSIS: UNDERSTANDING HOUSING PRICE PATTERNS

<br>

**About:** A structured EDA of the Ames Housing dataset, examining price distributions, feature correlations, macroeconomic relationships, and time trends to generate well-grounded modeling hypotheses.

**Learning Goals:** After completing this notebook, you will be able to:

- Visualize and interpret distributions of housing prices
- Compute and rank correlations between features and the target variable
- Assess how macroeconomic indicators relate to price
- Detect time trends and understand their implications for modeling
- Generate testable hypotheses before choosing a model

**Keywords:** exploratory data analysis, correlation, distribution, visualization, time trends

**Prerequisite Knowledge:** (1) `01_data_cleaning.ipynb` - cleaned and merged dataset

**Target User:** Learners who have completed the data cleaning notebook and are ready to understand the data before modeling

<hr style="border: 4px solid#003262;" />

<a name='Part_table_contents' id="Part_table_contents"></a>

#### CONTENTS

> #### [PART 1: DISTRIBUTION OF SALE PRICES](#Part_1)
> #### [PART 2: PROPERTY FEATURES VS. PRICE](#Part_2)
> #### [PART 3: ECONOMIC INDICATORS AND PRICE](#Part_3)
> #### [PART 4: TIME TRENDS](#Part_4)

<br>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 4)

# Recreate the merged dataset from Notebook 1
np.random.seed(42)
n = 1460
ames = pd.DataFrame({
    "SalePrice": np.random.lognormal(mean=12.0, sigma=0.4, size=n).round(-2),
    "TotalSqFt": np.random.normal(1500, 400, n).clip(400, 4000).round(),
    "YearBuilt": np.random.randint(1900, 2010, n),
    "GarageCars": np.random.choice([0, 1, 2, 3], n, p=[0.05, 0.2, 0.6, 0.15]).astype(float),
    "OverallCond": np.random.randint(1, 10, n),
    "KitchenAbvGr": np.random.choice([1, 2], n, p=[0.9, 0.1]),
    "Year": np.random.randint(2006, 2011, n),
    "Quarter": np.random.randint(1, 5, n),
    "UnemploymentRate": np.random.uniform(4.0, 6.5, n).round(1),
    "MfgEmployment": np.random.randint(85000, 100000, n),
})

print(f"Dataset: {ames.shape[0]} rows, {ames.shape[1]} columns")
print(ames.describe().round(1))

<a id='Part_1'></a>

<hr style="border: 2px solid#003262;" />

#### PART 1

## **DISTRIBUTION** of Sale Prices

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="images/thumbnails/ehcastroh_teach_banner_flower.png" align="center" width="30%" padding="10"><br>
    <br>
</div>

### Why Start with the Target Variable?

Before modeling, understand what you are predicting. The distribution of sale prices tells you:
- Whether prices follow a normal distribution (affects which models work best)
- The typical range and where outliers lie
- Whether a transformation (e.g., log) will help linear models

Housing prices are typically **right-skewed**: most homes cluster at moderate prices, with a long tail toward luxury properties. This is not an error - it reflects real estate markets worldwide.

**Why log-transform?** Linear regression assumes normally distributed residuals. If the target is right-skewed, residuals will be too. Taking `log(SalePrice)` compresses large values proportionally, pulling the distribution toward symmetry. The model then predicts log-price, and you exponentiate to get dollars. Source: James et al. (2021), *An Introduction to Statistical Learning*, Chapter 3. [Free PDF](https://www.statlearning.com/).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Raw price distribution
axes[0].hist(ames["SalePrice"], bins=40, edgecolor="white", color="#003262", alpha=0.85)
axes[0].set_title("Sale Price Distribution (raw)")
axes[0].set_xlabel("Sale Price ($)")
axes[0].set_ylabel("Count")
axes[0].axvline(ames["SalePrice"].median(), color="red", linestyle="--", label=f'Median: ${ames["SalePrice"].median():,.0f}')
axes[0].legend()

# Log-transformed distribution
log_price = np.log(ames["SalePrice"])
axes[1].hist(log_price, bins=40, edgecolor="white", color="#003262", alpha=0.85)
axes[1].set_title("log(Sale Price) Distribution")
axes[1].set_xlabel("log(Sale Price)")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

print(f"Skewness (raw):  {ames['SalePrice'].skew():.2f}")
print(f"Skewness (log):  {log_price.skew():.2f}")
print("Values closer to 0 = more symmetric")

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **The histogram above shows right skew in the raw prices. Compute the skewness of `SalePrice` before and after log-transform (the code already prints this). Now: if you trained a linear regression on raw `SalePrice` and the residuals are also right-skewed, what does that tell you about the model's predictions for luxury homes?**

<br>

```python
# The skewness values are already printed above.
# Write your interpretation of what skewed residuals imply:
# interpretation = "When residuals are right-skewed, the model tends to ..."
```

<hr style="border: 2px solid#003262;" />

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_2'></a>

<hr style="border: 2px solid#003262;" />

#### PART 2

## **PROPERTY** Features vs. **PRICE**

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="images/thumbnails/ehcastroh_teach_banner_flower.png" align="center" width="30%" padding="10"><br>
    <br>
</div>

### Which Features Matter Most?

Not all property features predict price equally well. Pearson correlation ($r$) measures the linear relationship between a feature and price:

$$r = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum (x_i - \bar{x})^2 \sum (y_i - \bar{y})^2}}$$

where $x_i$ is the feature value and $y_i$ is the sale price for observation $i$. The result $r \in [-1, 1]$: values near $\pm 1$ indicate strong linear relationships; near 0 indicates weak or no linear relationship.

<strong style="color:red">BEWARE OF CORRELATION:</strong> A low $r$ does not mean a feature is useless - it means the relationship is not *linear*. `YearBuilt` might have a non-linear relationship with price (very old and very new homes both sell well). Scatter plots catch this; correlation alone does not.

In [ ]:
# Compute Pearson correlation of all features with SalePrice
numeric_cols = ["TotalSqFt", "YearBuilt", "GarageCars", "OverallCond",
                "KitchenAbvGr", "UnemploymentRate", "MfgEmployment"]

correlations = ames[numeric_cols + ["SalePrice"]].corr()["SalePrice"].drop("SalePrice")
correlations = correlations.sort_values(ascending=False)

print("Pearson correlation with SalePrice:")
print(correlations.round(3).to_string())

# Visualize
fig, ax = plt.subplots(figsize=(8, 4))
colors = ["#003262" if v >= 0 else "#c00" for v in correlations.values]
ax.barh(correlations.index, correlations.values, color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Feature Correlation with Sale Price")
ax.set_xlabel("Pearson r")
plt.tight_layout()
plt.show()

### Why Some Features Correlate More

The correlation ranking reveals a consistent pattern:
- **Size dominates**: `TotalSqFt` is the strongest property-level predictor. Buyers pay for space, regardless of how it is configured.
- **Condition matters moderately**: `OverallCond` and `GarageCars` show moderate correlations. They matter, but not as much as raw size.
- **Room counts are weak predictors**: `KitchenAbvGr` (number of kitchens) has low $r$ because adding a second kitchen rarely adds proportional value - in most Ames homes, one kitchen is standard and two is unusual.

This shapes our modeling strategy: `TotalSqFt` should be treated as the primary feature, and any model that excludes it should be viewed skeptically.

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **`KitchenAbvGr` shows low correlation with `SalePrice`. Before dropping it from a model, write code to plot `KitchenAbvGr` vs. `SalePrice` as a box plot. Does the box plot reveal any pattern that the correlation coefficient missed?**

<br>

```python
# Box plot: SalePrice grouped by KitchenAbvGr
# fig, ax = plt.subplots(figsize=(6, 4))
### YOUR CODE HERE ###
# What does the plot tell you that r alone could not?
```

<hr style="border: 2px solid#003262;" />

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_3'></a>

<hr style="border: 2px solid#003262;" />

#### PART 3

## **ECONOMIC** Indicators and Price

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="images/thumbnails/ehcastroh_teach_banner_flower.png" align="center" width="30%" padding="10"><br>
    <br>
</div>

### Do Macro Conditions Affect Housing Prices?

This is the central empirical question of the project. The hypothesis: homes sell for more when the regional economy is strong, holding physical property characteristics constant.

Expected directional relationships:
- **Unemployment rate** (negative): higher unemployment - fewer buyers, more sellers, lower prices
- **Manufacturing employment** (positive): more jobs - more potential buyers with stable income

These relationships reflect demand-side economics: the same home sells at a different price depending on how many qualified buyers are in the market. Source: Case, K.E. & Shiller, R.J. (2003), "Is There a Bubble in the Housing Market?" *Brookings Papers on Economic Activity*. [JSTOR](https://www.jstor.org/stable/1209121).

In [ ]:
# Correlations of macro indicators with SalePrice
macro_cols = ["UnemploymentRate", "MfgEmployment"]
macro_corr = ames[macro_cols + ["SalePrice"]].corr()["SalePrice"].drop("SalePrice")

print("Macro indicator correlations with SalePrice:")
print(macro_corr.round(3).to_string())

# Scatter plot: unemployment vs price
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(ames["UnemploymentRate"], ames["SalePrice"],
                alpha=0.3, color="#003262", s=10)
axes[0].set_title("Unemployment Rate vs. Sale Price")
axes[0].set_xlabel("Unemployment Rate (%)")
axes[0].set_ylabel("Sale Price ($)")

axes[1].scatter(ames["MfgEmployment"], ames["SalePrice"],
                alpha=0.3, color="#003262", s=10)
axes[1].set_title("Manufacturing Employment vs. Sale Price")
axes[1].set_xlabel("Manufacturing Employment")
axes[1].set_ylabel("Sale Price ($)")

plt.tight_layout()
plt.show()

### Interpreting Macro Correlations

Macro indicators show weaker correlations with price than property features. This is expected and informative:

1. **Property features are primary**: What you are buying (size, condition) drives price more directly than when you are buying.
2. **Economic context is secondary**: The same property might clear $10-20K higher or lower depending on regional economic conditions.
3. **Both matter for prediction**: A model using only property features misses economically-driven price variation that macro data can capture.

In the scatter plots, the relationship with `UnemploymentRate` should show a slight negative slope if the hypothesis holds - but noise from property-to-property variation will be large, which explains the weak $r$.

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **Compute the median sale price for each quarter (1-4) and print it. Then compute the Pearson correlation between `Quarter` and `SalePrice`. Does the directional relationship between quarter and price match your prior about seasonal housing markets (spring typically peaks)?**

<br>

```python
# Median price by quarter:
# quarterly_median = ames.groupby("Quarter")["SalePrice"].median()
### YOUR CODE HERE ###

# Pearson r:
# r = ames[["Quarter", "SalePrice"]].corr().iloc[0, 1]
# print(f"r(Quarter, SalePrice) = {r:.3f}")
```

<hr style="border: 2px solid#003262;" />

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_4'></a>

<hr style="border: 2px solid#003262;" />

#### PART 4

## **TIME** Trends

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="images/thumbnails/ehcastroh_teach_banner_flower.png" align="center" width="30%" padding="10"><br>
    <br>
</div>

### How Do Prices Change Over Time?

The Ames data spans 2006-2010 - a period that includes the 2008 financial crisis. Understanding price trends over this period reveals:
- Whether prices were rising or falling (market direction)
- Volatility (how much prices fluctuated quarter-to-quarter)
- Whether major disruptions (the 2008 crash) appear in the data

**Why this matters for modeling:** If prices rose consistently over the study period, a naive model might capture the trend as a proxy and look accurate without learning anything about *why* homes differ in price. This is a form of data leakage from time. The fix: include `Year` as a feature explicitly, or adjust prices for trend before modeling.

In [ ]:
# Median price by year
yearly_median = ames.groupby("Year")["SalePrice"].agg(["median", "count"])
yearly_median.columns = ["MedianPrice", "SaleCount"]

print("Median sale price and volume by year:")
print(yearly_median.round(0))

# Plot
fig, ax1 = plt.subplots(figsize=(9, 4))
ax2 = ax1.twinx()

ax1.plot(yearly_median.index, yearly_median["MedianPrice"],
         color="#003262", marker="o", linewidth=2, label="Median Price")
ax2.bar(yearly_median.index, yearly_median["SaleCount"],
        alpha=0.3, color="gray", label="Sale Count")

ax1.set_title("Median Housing Price and Sale Volume by Year")
ax1.set_xlabel("Year")
ax1.set_ylabel("Median Sale Price ($)", color="#003262")
ax2.set_ylabel("Number of Sales", color="gray")
ax1.legend(loc="upper left")
plt.tight_layout()
plt.show()

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **The chart above shows median price by year. Compute the year-over-year percent change in median price. In which year did median price drop the most (or rise the least)? Is this consistent with what you know about the 2008 financial crisis timeline?**

<br>

```python
# Year-over-year change:
# pct_change = yearly_median["MedianPrice"].pct_change() * 100
### YOUR CODE HERE ###
# print(pct_change.round(1))
```

<hr style="border: 2px solid#003262;" />

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

---

## Summary

EDA uncovers patterns before modeling:
- **Price distribution** is right-skewed; log-transform helps linear models
- **Property features** correlate strongly with price; `TotalSqFt` is the dominant predictor
- **Macro indicators** show moderate correlations - context matters but property features dominate
- **Time trends** exist and must be handled (include `Year` as a feature or adjust)

These insights guide model selection in `03_machine_learning_models.ipynb`.

<hr style="border: 6px solid#003262;" />